In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=api_key,model="llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000023B9F7E9F30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023B9F7E9E40>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain.schema import (AIMessage,HumanMessage,SystemMessage)


ModuleNotFoundError: No module named 'langchain'

In [ ]:
##we have summarize this speech 
speech="""An Independence Day speech honors the sacrifices of freedom fighters for national independence, 
celebrates the nation's unity and values, and calls for active participation in building a strong, prosperous,
 and democratic future. Key elements include acknowledging the historical significance of August 15, 1947,
   recognizing prominent leaders and countless unnamed heroes, emphasizing the importance of national symbols
 like the flag and anthem, and pledging to uphold values of justice, liberty, equality, and fraternity. """

In [ ]:
speech

In [ ]:
chat_message=[
    SystemMessage(content="you are expert with expertise in summarizing speeches"),
    HumanMessage(content=f"please provide a short  and concise summary of the follow speech\n text:{speech}")
]

In [ ]:
import sys
print(sys.executable)


In [ ]:
import transformers
print("Transformers version:", transformers.__version__)


In [ ]:
!{sys.executable} -m pip install transformers


In [ ]:
import transformers
print("Transformers version:", transformers.__version__)


In [ ]:
llm.get_num_tokens(speech)  ##determine how many numbers of tokens present in the speech

In [ ]:
##chatmessage acting like a basic prompt
##here we are [parsing]  chat message
llm(chat_message).content

In [ ]:
# 1st method Geting the summary 

llm(chat_message)  ##Ai message is respnse from llm model  ---this is one way to summarize the data  
##here in output input token was 154 and output token we get is 71 hence the speech is summarized

In [ ]:
## 2nd method 
##prompt template_text _summarization


##prompt template_text _summarization

In [ ]:
from langchain.chains import LLMChain  ## whenever we combine llm and prompt template
##in case of llmchain we just use llm and promt template not strouputparser
  ##here convert generic template into promt
from langchain import PromptTemplate

generic_template="""write the summary of the following speech :
speech:{speech}
tranlate the precide summary to {language}"""  

prompt=PromptTemplate(
    input_variables=['speech','language'],
    template=generic_template
)
prompt

In [ ]:
##to see entire prompt
complete_prompt=prompt.format(speech=speech,language="French")
complete_prompt

In [ ]:
llm.get_num_tokens(complete_prompt)  ## why no.of tokens get increased bcz we have given additional info. in generric template along with speech 

In [ ]:
##if you have llm amd prompt then use llmchain
llm_chain=LLMChain(llm=llm,prompt=prompt)
summary=llm_chain.run({'speech':speech,'language':'english'})  #use any language
summary

In [ ]:
##here we have limited text but in case of huge text we need to use another techniques



(for large data)

StuffDocumentChain Text Summarization--which simply concatenates documents into a prompt

MapReduce ----which splits documents into batches ,summarizes those and then summarizes the summaries(for larger files)----it has two important types 
a) single prompt template
b) multiple prompt template
Refine ------which updates a rolling summary  by iterating over the documents in a sequence

StuffDocumentChain Text Summarization--- it is very basic type of text summarization 



In [ ]:
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("SIH.pdf")
docs=loader.load_and_split()
docs  ## here we get so many documents in output

In [ ]:
##let's create prompt template
template=""" write a concise and short summary of the following speech,
specch:{text}"""

prompt=PromptTemplate(input_variable=['text'],template=template)


In [ ]:
##for text summarization we use this chain given below
from langchain.chains.summarize import load_summarize_chain
chain=load_summarize_chain(llm,chain_type="stuff",prompt=prompt,verbose=True) 
output_summary=chain.run(docs)
output_summary

## Map Reduce to Summarize Large Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs

In [ ]:
final_documents=RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=100).split_documents(docs)
final_documents  ## now it will take each documnet and then summarize it

In [ ]:
len(final_documents)

In [ ]:
##now lets create prompt template  (this is for each document)
chunk_prompt="""
please summarize the below speech:
speech:'{text}'
Summary:
"""

map_prompt_templates=PromptTemplate(input_variable=['text'],template=chunk_prompt)

In [ ]:
## this is final prompt template
final_prompt='''
provide the final  summary  of the entire speech with these important points.
Add  a motivational title ,start  the precise summary with an introduction and provide the summary in number points for the speech
speech:{text}
'''   ##here i want entire summary in the form of number

final_prompt_template=PromptTemplate(input_variable=['text'],template=final_prompt)
final_prompt_template

In [ ]:
summarize_chain=load_summarize_chain(
llm=llm,
chain_type="map_reduce",
map_prompt=map_prompt_templates,   ## this will give the summary of smaller chunks
combine_prompt=final_prompt_template,
verbose=True     ## combine smaller chunk  summary with this
)

output=summarize_chain.run(final_documents)
output

##  Refine chain summarization 


In [ ]:
chain=load_summarize_chain(
    llm=llm,
    chain_type="refine",
    verbose=True,
)

output_summary=chain.run(final_documents)
output_summary

verbose=True → The program will print messages or show progress, like what step it’s doing, loading data, generating output, etc.

verbose=False → The program will stay quiet and not show details, only giving the final result.


Example:

from langchain import LLMChain

chain = LLMChain(llm=llm, prompt=prompt, verbose=True)

Here, if verbose=True, you’ll see something like:

Running chain...
Sending prompt to LLM...
Received response: "The answer is 42"

If it was verbose=False, you’d just get:

"The answer is 42"

So, verbose=True helps you understand what’s happening step by step — great for debugging or learning.